# Análisis de Datos · Semana 15, sesión 1 de 3
## Series, DataFrame y carga de archivos

**TIA502 · Facultad de Empresariales · Profesor David Escobar-Castillejos**

La hoja de cálculo que ya sabes usar existe dentro de Python y se llama DataFrame. Esta
sesión presenta los dos objetos sobre los que está construido todo pandas, y una costumbre
de dos minutos que separa un análisis correcto de uno que solo se ve correcto.

Al terminar este cuaderno vas a poder:

1. Explicar qué es una `Series` y por qué es la lista de la semana 12 con etiquetas encima.
2. Construir un `DataFrame` desde un diccionario de columnas y desde un archivo.
3. Cargar un CSV en una línea con `read_csv`, y saber qué hizo con cada columna.
4. Inspeccionar un archivo con `head`, `info`, `shape` y `describe`, siempre en ese orden.
5. Detectar valores faltantes, renglones duplicados y categorías mal capturadas antes de
   que arruinen un resultado.

### Cómo se usa este cuaderno

Ejecuta las celdas en orden, de arriba hacia abajo. Varias dependen de una variable que
definió la anterior, así que saltarte una da un `NameError` que no tiene nada que ver con
el tema.

Antes de correr una celda marcada **Predice**, escribe tu respuesta en un papel. Fallar la
predicción y entender por qué enseña más que ver la salida correcta de primera.

Tres celdas fallan a propósito. Llevan un comentario que lo dice y atrapan el error para
que el cuaderno siga corriendo.

---
## Preparación

Dos celdas de arranque. La primera dice qué versión de pandas te tocó, la segunda deja los
datos a la mano.

In [ ]:
import pandas as pd

print("pandas", pd.__version__)

### Qué esperar según la versión

Colab actualiza sus bibliotecas cuando quiere, así que el número de arriba puede ser 2.x o
3.x. Importa para una sola cosa en esta sesión: cómo se llama el tipo de una columna de
texto.

| | pandas 2.x | pandas 3.0 y posteriores |
|---|---|---|
| Una columna de texto reporta | `object` | `str` |
| `info()` cierra con | `dtypes: float64(1), object(5)` | `dtypes: float64(1), str(5)` |

Es el mismo dato y el mismo comportamiento, con otro nombre. `object` era el cajón donde
pandas guardaba cualquier cosa que no fuera número; desde la versión 3.0 el texto tiene su
propio tipo y ya no comparte cajón con nadie. Si tu salida dice `object` donde este
cuaderno dice `str`, no te equivocaste en nada.

In [ ]:
# Plomería, no lección. Esta celda deja los tres CSV del curso al
# alcance de pandas y no vuelve a hacer falta.
#
# Primero los busca en el repositorio, que es público y se lee por URL.
# Si no responde, los reconstruye aquí mismo con la semilla fija del
# curso, así que salen idénticos por cualquiera de los dos caminos.
# En ningún caso hay que subir un archivo a mano.
import urllib.request
from pathlib import Path

BASE = ("https://raw.githubusercontent.com/Davidowa/learning-hub/main/"
        "docs/en/courses/python-course/06%20-%20Advanced/data/")
ARCHIVOS = ["sales.csv", "regions.csv", "employees.csv"]


def _descargar():
    for nombre in ARCHIVOS:
        with urllib.request.urlopen(BASE + nombre, timeout=15) as r:
            Path(nombre).write_bytes(r.read())


def _reconstruir_datos():
    """Vuelve a escribir los tres CSV con la semilla fija del curso.

    Salen idénticos byte por byte a los del repositorio, así que los
    números de la diapositiva siguen coincidiendo con los del cuaderno.
    """
    import csv, random
    from datetime import date, timedelta

    rng = random.Random(20260808)
    REGIONS = ["North", "South", "Centre", "West"]
    CHANNELS = ["Retail", "Online", "Wholesale"]
    PRODUCTS = {"Espresso machine": 8990.0, "Coffee grinder": 2450.0,
                "Filter kettle": 1290.0, "Bean subscription": 690.0,
                "Travel mug": 349.0}
    RW = {"North": 1.30, "South": 0.80, "Centre": 1.55, "West": 0.95}
    CW = {"Retail": 1.00, "Online": 1.25, "Wholesale": 2.10}
    MW = [0.72, 0.78, 0.90, 0.95, 1.00, 1.05, 0.98, 0.92, 1.08, 1.15, 1.45, 1.60]

    rows, start = [], date(2025, 1, 6)
    for week in range(52):
        day = start + timedelta(weeks=week)
        for region in REGIONS:
            for _ in range(rng.randint(1, 2)):
                product = rng.choice(list(PRODUCTS))
                channel = rng.choice(CHANNELS)
                base = 9 * RW[region] * CW[channel] * MW[day.month - 1]
                units = max(1, round(rng.gauss(base, base * 0.28)))
                price = PRODUCTS[product] * rng.choice([1.0, 1.0, 1.0, 0.9, 0.85])
                rows.append({"date": day.isoformat(), "region": region,
                             "channel": channel, "product": product,
                             "units": str(units),
                             "unit_price": f"$ {price:,.2f}"})

    # la suciedad deliberada: una región tecleada de cuatro formas,
    # celdas en blanco, y renglones capturados dos veces
    for i in rng.sample(range(len(rows)), 24):
        rows[i]["region"] = rng.choice(["north", "NORTH", " North", "North "])
    for i in rng.sample(range(len(rows)), 11):
        rows[i]["units"] = ""
    for i in rng.sample(range(len(rows)), 7):
        rows.append(dict(rows[i]))
    rng.shuffle(rows)

    with open("sales.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["date", "region", "channel",
                                          "product", "units", "unit_price"])
        w.writeheader()
        w.writerows(rows)

    with open("regions.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["region", "manager", "country", "monthly_target"])
        w.writerows([["North", "Ana Robles", "Mexico", 480000],
                     ["South", "Luis Ferrer", "Mexico", 300000],
                     ["Centre", "Paula Ines", "Mexico", 560000],
                     ["West", "Marco Duarte", "Mexico", 360000],
                     ["East", "Sofia Lara", "Mexico", 220000]])

    AREAS = {
        "Sales": (["Account executive", "Sales analyst", "Sales manager"], 24000, 62000),
        "Marketing": (["Content specialist", "Campaign analyst", "Brand manager"], 22000, 58000),
        "Finance": (["Accounts clerk", "Financial analyst", "Controller"], 26000, 74000),
        "People": (["Recruiter", "People analyst", "People manager"], 21000, 55000),
        "Operations": (["Warehouse lead", "Logistics analyst", "Operations manager"], 20000, 60000),
    }
    CITIES = ["Mexico City", "Guadalajara", "Monterrey", "Queretaro"]
    emp = []
    for n in range(1, 121):
        area = rng.choice(list(AREAS))
        titles, low, high = AREAS[area]
        idx = rng.choices([0, 1, 2], weights=[5, 3, 1])[0]
        tenure = rng.randint(2, 132)
        salary = round(low + (high - low) * (idx / 2) * rng.uniform(0.82, 1.10)
                       + tenure * 45, -2)
        emp.append({"employee_id": f"E{n:04d}", "area": area,
                    "job_title": titles[idx], "city": rng.choice(CITIES),
                    "tenure_months": tenure, "monthly_salary": int(salary)})
    with open("employees.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(emp[0]))
        w.writeheader()
        w.writerows(emp)


try:
    _descargar()
    print("Datos leídos del repositorio.")
except Exception:
    _reconstruir_datos()
    print("El repositorio no respondió. Datos reconstruidos en esta sesión.")

print("Listos:", ", ".join(ARCHIVOS))

---
# Bloque 1 · Los dos objetos

Todo pandas está construido sobre dos cosas, y las dos ya las conoces con otro nombre. Una
es la columna. La otra es la hoja completa.

| En la hoja | En pandas | Lo que ya sabías |
|---|---|---|
| Una columna | `Series` | Una lista, con índice |
| La hoja completa | `DataFrame` | Varias listas emparejadas |
| El número de fila | El índice | La posición, que empieza en 0 |
| El encabezado | `columns` | Las llaves de un diccionario |

La correspondencia es exacta, y esa es la razón por la que este curso puede saltar directo
a pandas sin pasar por una biblioteca intermedia.

## Una Series es una columna

La forma más simple de hacer una es desde una lista.

In [ ]:
unidades = pd.Series([15, 8, 22, 5, 11])
print(unidades)

Regresaron dos cosas que la lista no tenía.

A la izquierda apareció un **índice**, del 0 al 4, que pandas creó solo. Abajo apareció un
**dtype**, el tipo que comparten todos los valores de la columna. Una lista de Python puede
mezclar enteros con texto; una Series no, y de ahí sale casi toda su velocidad.

In [ ]:
print("El índice:", list(unidades.index))
print("El dtype: ", unidades.dtype)
print("El largo: ", len(unidades))

El índice no tiene que ser un contador. Ponle etiquetas y la Series empieza a comportarse
como un rango con nombre de tu hoja de cálculo.

In [ ]:
mensual = pd.Series(
    [42000, 51500, 38900, 60100, 55300, 47800],
    index=["ene", "feb", "mar", "abr", "may", "jun"],
)
print(mensual)

Ahora un valor se alcanza por su etiqueta y no contando posiciones. Es la diferencia entre
escribir `mensual["mar"]` y acordarte de que marzo era el tercero, o el segundo si cuentas
desde cero.

In [ ]:
print("Marzo:", mensual["mar"])
print("Los tres primeros meses:")
print(mensual[["ene", "feb", "mar"]])

Las funciones de resumen que usas en la hoja son métodos de la Series. Son las mismas
cuentas, escritas de otra forma.

In [ ]:
print("Total:     ", mensual.sum())
print("Promedio:  ", round(mensual.mean(), 2))
print("Mejor mes: ", mensual.idxmax(), "con", mensual.max())
print("Peor mes:  ", mensual.idxmin(), "con", mensual.min())

### El cambio grande: la operación se aplica a la columna entera

Una operación sobre una Series alcanza a todos los valores de golpe. No hay `for`, y no hay
que arrastrar la fórmula hacia abajo. Este es el cambio más grande en cómo vas a trabajar
de aquí en adelante.

In [ ]:
con_iva = mensual * 1.16
print(con_iva.round(2))

La comparación funciona igual, y devuelve una Series de verdaderos y falsos. Esa Series de
booleanos es lo que después va a servir para filtrar renglones, así que vale la pena verla
sola antes de usarla dentro de un corchete.

In [ ]:
buenos = mensual > 50000
print(buenos)
print()
print("Meses arriba de 50 mil:", buenos.sum())
print(mensual[buenos])

### El caso límite: dos Series con índices distintos

Aquí es donde el índice deja de ser un adorno. Cuando sumas dos Series, pandas no empareja
por posición sino por etiqueta. Si una etiqueta existe en una sola de las dos, el resultado
en esa fila es `NaN`, que es como pandas escribe "no hay dato".

**Predice antes de correr.** La primera Series tiene enero, febrero y marzo. La segunda
tiene febrero, marzo y abril. ¿Cuántas filas trae el resultado y qué hay en cada una?

In [ ]:
primer = pd.Series([100, 200, 300], index=["ene", "feb", "mar"])
segundo = pd.Series([10, 20, 30], index=["feb", "mar", "abr"])

print(primer + segundo)

Salieron cuatro filas y no tres, porque pandas conservó la unión de las dos etiquetas.
Enero y abril quedaron en `NaN` porque a cada uno le faltó su pareja. Febrero y marzo sí se
sumaron.

Esto es correcto y es lo que quieres casi siempre, pero sorprende la primera vez. Si
esperabas tres números y te salieron cuatro con dos huecos, el índice es el culpable, no la
suma.

## Un DataFrame son varias Series que comparten el índice

Lo normal es construirlo desde un diccionario, con una llave por columna.

In [ ]:
ventas_demo = pd.DataFrame({
    "mes": ["ene", "feb", "mar", "abr", "may", "jun"],
    "region": ["North", "North", "South", "South", "Centre", "Centre"],
    "monto": [42000, 51500, 38900, 60100, 55300, 47800],
    "unidades": [15, 18, 12, 21, 19, 16],
})
print(ventas_demo)

In [ ]:
print("Forma (renglones, columnas):", ventas_demo.shape)
print("Nombres de columna:", list(ventas_demo.columns))
print("Índice:", list(ventas_demo.index))

Cada columna es una Series, y se saca por nombre. Que el tipo sea literalmente `Series` es
lo que hace que todo lo del bloque anterior siga sirviendo aquí.

In [ ]:
print(ventas_demo["monto"])
print()
print("Su tipo:", type(ventas_demo["monto"]))

In [ ]:
print(ventas_demo.dtypes)

Los `dtypes` te dicen cómo entendió pandas cada columna. `mes` y `region` traen texto,
`monto` y `unidades` traen enteros. Este ejemplo se armó a mano y por eso salió limpio; en
un archivo real es donde empiezan las sorpresas, y de eso trata el bloque 3.

### Una columna nueva a partir de las otras

Asignar a un nombre que no existe crea la columna. El lado derecho se calcula para todos
los renglones a la vez, que es exactamente lo que hace una fórmula arrastrada hasta abajo.

In [ ]:
ventas_demo["precio_unitario"] = (ventas_demo["monto"] / ventas_demo["unidades"]).round(2)
print(ventas_demo)

Y una variante que la diapositiva no alcanzó a mostrar: la columna nueva puede salir de una
comparación, no solo de una división. Aquí marca los meses que pasaron de cincuenta mil.

In [ ]:
ventas_demo["buen_mes"] = ventas_demo["monto"] > 50000
print(ventas_demo[["mes", "monto", "buen_mes"]])
print()
print("Cuántos buenos:", ventas_demo["buen_mes"].sum())

---
# Bloque 2 · Cargar el archivo

La semana pasada abriste un CSV a mano con el módulo `csv`: abrir el archivo, leer el
encabezado, recorrer los renglones y convertir cada campo. `read_csv` hace todo eso en una
línea.

Vale la pena correr las dos versiones seguidas, porque la comparación es el argumento
entero a favor de pandas.

## La versión a mano, la de la semana 14

In [ ]:
import csv

with open("sales.csv", encoding="utf-8") as f:
    filas = list(csv.DictReader(f))

for fila in filas:
    fila["units"] = int(fila["units"] or 0)

print(len(filas), "renglones")
print(filas[0])

Funciona, y cada renglón quedó como un diccionario. El problema no es que sea largo, es que
tú tomaste una decisión escondida: `int(fila["units"] or 0)` convirtió las celdas vacías en
cero. Eso cambia el promedio y nadie lo va a notar leyendo el código.

## La versión con pandas

In [ ]:
ventas = pd.read_csv("sales.csv")

print(ventas.shape)

Una línea, y además `read_csv` intentó adivinar el tipo de cada columna. Lo que no puede
adivinar es qué querías que significara una celda vacía, y por eso no la rellena con cero:
la deja marcada como faltante y te deja a ti decidir. Esa decisión es la de la sesión 15.2.

### Un detalle que sirve todo el semestre

`read_csv` acepta una URL donde acepta una ruta. Con el repositorio público, cargar los
datos del curso desde cualquier máquina se ve así, sin descargar nada a mano:

```python
BASE = ("https://raw.githubusercontent.com/Davidowa/learning-hub/main/"
        "docs/en/courses/python-course/06%20-%20Advanced/data/")

ventas = pd.read_csv(BASE + "sales.csv")
```

La celda de preparación de este cuaderno intenta exactamente eso antes de reconstruir los
archivos por su cuenta.

## Los primeros renglones

In [ ]:
print(ventas.head())

`head` muestra los primeros cinco renglones. Pásale un número para cambiar cuántos.

In [ ]:
print(ventas.head(3))

`tail` muestra los últimos, y es la que hay que correr aunque parezca redundante. Ahí es
donde se esconde el renglón de totales que alguien pegó al final de la hoja antes de
exportarla, y que si no lo ves acaba contado como una venta más.

In [ ]:
print(ventas.tail(3))

`shape` responde cuántos datos tienes de verdad. Devuelve una tupla, así que se puede
desempacar en dos nombres.

In [ ]:
renglones, columnas = ventas.shape
print(f"{renglones} renglones y {columnas} columnas")

---
# Bloque 3 · Mirar antes de tocar

La tentación es ir directo a la respuesta. Aguántate dos minutos. Casi todo resultado
equivocado del proyecto final se rastrea a una columna que no era del tipo que supusiste, o
a renglones que no estaban.

El orden es siempre el mismo: `head`, `info`, `shape`, `describe`.

## Los tipos que infirió

In [ ]:
print(ventas.dtypes)

`info` es el comando más útil de esta sesión. Reporta, por columna, cuántos valores no
están vacíos y qué tipo infirió pandas. Fíjate en la columna `Non-Null Count`: una que no
llega al total de renglones tiene huecos.

In [ ]:
ventas.info()

### Qué te están diciendo esos tipos

Tres cosas de esa salida merecen nombre propio.

**`units` entró como `float64` y no como entero.** pandas leyó bien los dígitos, pero once
celdas están vacías, y un vacío se tiene que representar de alguna forma. Ese marcador es
`NaN`, que solo existe en una columna decimal, así que la columna entera se volvió decimal.
Por eso los conteos se imprimen como `15.0` en lugar de `15`.

**`unit_price` entró como texto**, porque `"$ 2,082.50"` no es un número para Python. El
signo de pesos y la coma de miles son formato, y el formato no es parte del valor.

**`date` entró como texto**, porque un CSV no tiene tipo fecha. Mientras no se convierta,
ordenar por esa columna funciona de milagro: sale bien solo porque el formato pone el año
primero.

Ninguna de las tres es una falla de pandas. Es el CSV, que no guarda tipos, exactamente
como lo viste la semana pasada. La sesión 15.2 arregla las tres.

## El resumen numérico

In [ ]:
print(ventas.describe())

`describe` da conteo, promedio, desviación estándar, mínimo, máximo y los cuartiles de cada
columna numérica. Por ahora solo `units` califica, y su conteo de 313 contra 324 renglones
es otra vez el dato faltante asomándose.

Esa es la razón por la que `describe` en un archivo recién cargado dice tan poco: no es que
no haya números, es que están guardados como texto. Con `include="all"` pandas también
resume las columnas de texto, con otras estadísticas.

In [ ]:
print(ventas.describe(include="all"))

Dos renglones de esa tabla valen la revisada. `unique` cuenta cuántos valores distintos hay
en la columna, y `top` dice cuál se repite más. `region` reporta ocho valores distintos, y
en la empresa hay cuatro regiones.

## Encontrar la suciedad

Para una columna de texto, lo que sirve es contar cuántas veces aparece cada valor.

In [ ]:
print(ventas["region"].value_counts())

Se capturaron cuatro regiones y el archivo cree que hay ocho, porque el mismo nombre se
escribió con distinta capitalización y con espacios de sobra. `" North"`, `"North "`,
`"north"` y `"NORTH"` son, para Python, cuatro textos que no tienen nada que ver entre sí.

Si agrupas por región hoy, el norte se te parte en cinco pedazos y ninguno trae el total
verdadero. Esto se arregla en la sesión 15.2, y el punto de hoy es que se detecta antes de
que pase.

In [ ]:
print("Valores distintos en region:", ventas["region"].nunique())
print("Valores distintos en channel:", ventas["channel"].nunique())
print("Valores distintos en product:", ventas["product"].nunique())

`isna` marca cada celda faltante como verdadero, y `sum` las cuenta por columna.

In [ ]:
print(ventas.isna().sum())
print()
print("Faltantes en toda la tabla:", ventas.isna().sum().sum())

`duplicated` marca un renglón como verdadero cuando un renglón idéntico ya apareció antes.
Los siete de aquí son el rastro de un copiar y pegar.

In [ ]:
print("Renglones duplicados:", ventas.duplicated().sum())
print()
print(ventas[ventas.duplicated(keep=False)].sort_values("date").head(6))

Nota el `keep=False` de la última línea: con ese argumento pandas marca todas las copias,
no solo las repeticiones, así que puedes ver los pares completos y confirmar que de verdad
son idénticos.

Y ojo con la aritmética. 324 renglones con siete duplicados son 317 hechos distintos. Cuál
de los dos números va en tu reporte depende de qué estés contando, y decidirlo es tu
trabajo, no el de pandas.

---
## Tres celdas que fallan a propósito

Un tipo equivocado no siempre truena. A veces da un número, y ese es el caso peligroso.

### La que no falla, y por eso es la peor

In [ ]:
# FALLA A PROPÓSITO. Esta celda no lanza ningún error, y ese es justo el problema.
# unit_price es texto, así que sum() concatena en lugar de sumar.
total = ventas["unit_price"].sum()

print("Tipo del resultado:", type(total))
print("Primeros 70 caracteres:", str(total)[:70])

Pediste un total de ventas y recibiste los 324 precios pegados uno tras otro en un solo
texto. Ningún error, ninguna advertencia. Si esto va dentro de un reporte más largo, sale
publicado.

Este es el argumento entero a favor de correr `info()` antes de analizar. Dos minutos de
inspección contra un número mal en una presentación.

### La que sí falla, y avisa a tiempo

In [ ]:
# FALLA A PROPÓSITO. Promediar texto sí lanza error, a diferencia de sumarlo.
try:
    ventas["unit_price"].mean()
except TypeError as e:
    print("TypeError:", e)

`mean` no tiene forma de inventar un promedio de textos, así que se detiene. Es el mismo
problema que la celda anterior, con mejor suerte: aquí el error aparece cuando lo
escribiste, y no tres semanas después en una junta.

### La conversión que parece obvia y no lo es

In [ ]:
# FALLA A PROPÓSITO. units trae once NaN, y NaN no cabe en un entero.
try:
    ventas["units"].astype(int)
except Exception as e:
    print(type(e).__name__ + ":", e)

Si te molestó ver `15.0` donde esperabas `15`, este es el intento natural de arreglarlo, y
no funciona. No puedes convertir a entero mientras haya faltantes, porque `NaN` no es un
número entero representable.

El orden correcto es al revés de como se siente: primero decides qué significan los once
huecos, después conviertes. Rellenar con cero, descartar esos renglones o dejarlos como
faltantes son tres decisiones distintas con tres promedios distintos, y ninguna es la
predeterminada.

## La segunda tabla

In [ ]:
regiones = pd.read_csv("regions.csv")
print(regiones)

Esta es la tabla de consulta del curso, el equivalente del `BUSCARV` de tu hoja: un renglón
por región, con los datos que no tienen por qué repetirse en cada venta.

Trae cinco regiones y el archivo de ventas solo cubre cuatro. La diferencia es
intencional, y la sesión 15.3 muestra qué hace una unión con la región que se quedó sin
ventas.

---
## Predice antes de correr

Escribe tu respuesta antes de ejecutar cada celda.

### Pregunta 1

¿Por qué la columna `units` salió `float64` y no `int64`?

- **A.** Porque las unidades traen decimales en el archivo.
- **B.** Porque once celdas vacías necesitan `NaN`, que solo existe en float.
- **C.** Porque `read_csv` siempre usa float por seguridad.
- **D.** Porque la columna tiene más de trescientos renglones.

In [ ]:
print("dtype de units:      ", ventas["units"].dtype)
print("Faltantes en units:  ", ventas["units"].isna().sum())
print("¿Algún valor con parte decimal?")
print((ventas["units"].dropna() % 1 != 0).sum(), "de", ventas["units"].notna().sum())

Cero valores con parte decimal, y once faltantes. La respuesta es **B**: los números eran
enteros desde el principio, y lo que forzó el tipo decimal fue el marcador de vacío.

### Pregunta 2

`ventas.shape` dijo 324 renglones. ¿Cuántas ventas distintas describe el archivo, y cuántas
tienen su conteo de unidades registrado?

In [ ]:
print("Renglones:                   ", len(ventas))
print("Sin contar duplicados:       ", len(ventas.drop_duplicates()))
print("Con units registrado:        ", ventas["units"].notna().sum())
print("Distintos y con units:       ", len(ventas.drop_duplicates().dropna(subset=["units"])))

Cuatro números distintos, todos correctos, todos respuesta a preguntas diferentes. Cuál usar
depende de qué afirmes en el reporte, y por eso `shape` no es "el número de datos" sino el
primero de varios.

### Pregunta 3

¿Qué imprime la siguiente celda? Piensa en qué tipo tiene `ventas["units"]` y qué tipo
tiene `ventas[["units"]]`.

In [ ]:
print(type(ventas["units"]))
print(type(ventas[["units"]]))
print()
print(ventas[["date", "region", "units"]].head(3))

Un corchete devuelve una `Series`, la columna sola. Dos corchetes devuelven un `DataFrame`,
porque lo que pasaste fue una lista de nombres y una lista puede traer más de uno. Es la
confusión más común de las próximas dos sesiones, y ahora ya la viste con sus dos tipos
impresos.

---
## Cuatro errores al cargar un archivo

**Analizar antes de inspeccionar.** El resultado sale, se ve razonable y está mal. `head`,
`info` y `describe` cuestan dos minutos y son la única defensa contra el error que nadie
encuentra porque nadie lo está buscando.

**Confiar en el tipo inferido.** pandas adivina bien casi siempre. Ese *casi* es donde vive
la columna de precios que resultó ser texto.

**No revisar las categorías.** `value_counts` sobre una columna de texto delata la captura
inconsistente antes de que te parta los grupos en cinco pedazos.

**Suponer que `shape` es el número de datos.** 324 renglones con siete duplicados son 317
hechos distintos, y el total cambia según cuál cuentes.

---
# Ejercicios

Resuélvelos en celdas nuevas debajo de cada enunciado. Las soluciones están hasta abajo del
cuaderno, así que no las alcanzas a ver de reojo mientras trabajas.

Van de menos a más. Los primeros cuatro repiten con otros datos lo que acabas de ver; los
tres siguientes te piden combinar dos ideas; el último es sobre tu propio archivo.

## Para calentar

### Ejercicio 1 · Una Series con etiquetas

Arma una `Series` con las ventas de los seis primeros meses del año, usando las etiquetas
de mes como índice. Imprime el total, el promedio redondeado a dos decimales, y el nombre
del mes más flojo. Después súbele 8 % a todos los meses de una sola operación.

### Ejercicio 2 · Un DataFrame desde cero

Construye un `DataFrame` con cinco productos de una cafetería: nombre, precio unitario y
piezas vendidas en la semana. Agrega una columna `ingreso` que multiplique precio por
piezas, imprime la tabla ordenada de mayor a menor ingreso, y di cuánto se vendió en total.

Pista: `df.sort_values("ingreso", ascending=False)`.

### Ejercicio 3 · El diagnóstico completo

Escribe una función `diagnosticar(df)` que reciba un DataFrame e imprima, en este orden:
cuántos renglones y columnas tiene, qué tipo tiene cada columna, cuántos valores faltan por
columna, y cuántos renglones duplicados hay. Pruébala con `ventas` y con `regiones`.

### Ejercicio 4 · Las columnas de texto

Recorre las columnas de `ventas` y, para cada una que haya salido de tipo texto, imprime el
nombre y cuántos valores distintos tiene. Después di, en un comentario, cuáles de esas
columnas deberían convertirse a otro tipo y cuáles están bien como texto.

## Para pensarle

### Ejercicio 5 · Cuánto cuesta el dato faltante

Los once renglones sin unidades tienen tres destinos posibles: rellenarlos con cero,
descartarlos, o dejarlos como están. Calcula el promedio de `units` bajo las tres
decisiones y ponlos en la misma salida.

Después contesta en un comentario cuál usarías si el reporte dice "promedio de unidades por
venta", y por qué las otras dos estarían mal ahí.

Pistas: `.fillna(0)`, `.dropna()`, y `.mean()` que por su cuenta ya ignora los faltantes.

### Ejercicio 6 · El tamaño del desastre

Sin limpiar nada todavía, mide cuánto daño haría analizar el archivo tal como está. Cuenta
cuántos renglones traen una versión sucia de `"North"`, o sea cualquier valor de `region`
que no sea exactamente uno de los cuatro nombres correctos.

Después imprime, lado a lado, cuántos renglones cree el archivo que son del norte y cuántos
son de verdad.

Pista: `~ventas["region"].isin([...])` invierte una pertenencia.

### Ejercicio 7 · El empleado más caro por área

Carga `employees.csv`, que trae 120 renglones y sale limpio. Imprime cuántas áreas hay,
cuántas personas tiene cada una, y el salario mensual más alto de la tabla junto con el
identificador de quien lo cobra.

No necesitas agrupar todavía, eso es la sesión 15.3. Con `value_counts`, `max` e `idxmax`
alcanza, y ese es justo el punto del ejercicio.

## Con tus datos

### Ejercicio 8 · Tu propio archivo

Carga con pandas el CSV de tu proyecto y escribe un diagnóstico de media cuartilla que
cubra cuántos renglones tiene, qué tipo infirió cada columna, cuántos valores faltan y
cuántos duplicados hay.

Todavía no limpies nada. Hoy solo se mira y se anota. Por cada columna que salió de tipo
texto, di si eso está bien o si algo hay que convertir.

---
## Tres ideas para llevarse

**Una Series es una columna con índice.** Es la lista de la semana 12 con etiquetas, y todo
lo que aprendiste ahí sigue valiendo aquí. El índice no es adorno: es lo que empareja los
datos cuando combinas dos objetos.

**`read_csv` no adivina lo que no puede.** Infiere tipos bien casi siempre, y no tiene forma
de saber qué querías que significara una celda vacía. Esa decisión es tuya y cambia el
resultado.

**Inspeccionar antes de analizar.** `head`, `info`, `shape` y `describe`. Dos minutos que
evitan un resultado equivocado que se ve razonable, que es la peor clase de resultado
equivocado.

La siguiente sesión es seleccionar, filtrar y limpiar. Ahí se arregla todo lo que hoy
diagnosticamos.

---
# Soluciones

Compáralas con lo tuyo después de intentarlo. Si tu versión llega al mismo resultado por
otro camino, está bien: aquí no hay una sola forma correcta.

### Ejercicio 1

```python
ventas_mes = pd.Series(
    [42000, 51500, 38900, 60100, 55300, 47800],
    index=["ene", "feb", "mar", "abr", "may", "jun"],
)

print("Total:", ventas_mes.sum())
print("Promedio:", round(ventas_mes.mean(), 2))
print("Mes más flojo:", ventas_mes.idxmin(), "con", ventas_mes.min())

con_aumento = ventas_mes * 1.08
print(con_aumento.round(2))
```

El aumento se aplica a los seis meses en una sola línea. No hace falta un ciclo, y escribir
uno aquí es la señal más común de que alguien sigue pensando en listas.

### Ejercicio 2

```python
cafe = pd.DataFrame({
    "producto": ["Americano", "Capuchino", "Latte", "Concha", "Croissant"],
    "precio": [38.0, 52.0, 55.0, 24.0, 46.0],
    "piezas": [310, 185, 142, 260, 98],
})

cafe["ingreso"] = cafe["precio"] * cafe["piezas"]
print(cafe.sort_values("ingreso", ascending=False))
print("\nIngreso total:", cafe["ingreso"].sum())
```

`sort_values` devuelve una tabla nueva y deja la original intacta. Si querías que el cambio
se quedara, hay que reasignar: `cafe = cafe.sort_values(...)`. Casi todos los métodos de
pandas se comportan así, y esa es una de las razones por las que la sesión 15.2 empieza
hablando de copias.

### Ejercicio 3

```python
def diagnosticar(df):
    renglones, columnas = df.shape
    print(f"{renglones} renglones y {columnas} columnas")

    print("\nTipos por columna:")
    print(df.dtypes)

    print("\nFaltantes por columna:")
    print(df.isna().sum())

    print("\nRenglones duplicados:", df.duplicated().sum())


diagnosticar(ventas)
print("\n" + "=" * 40 + "\n")
diagnosticar(regiones)
```

`regiones` sale limpio: cinco renglones, sin faltantes, sin duplicados. Ese contraste es
útil, porque enseña cómo se ve un archivo sano y te da con qué comparar.

### Ejercicio 4

```python
for nombre in ventas.columns:
    if ventas[nombre].dtype == "object" or ventas[nombre].dtype == "str":
        print(f"{nombre:12} {ventas[nombre].nunique():4} valores distintos")

# date        debería convertirse a fecha, para poder ordenar y agrupar por mes
# region      debería normalizarse a cuatro valores, no convertirse de tipo
# channel     está bien como texto, tres valores y todos consistentes
# product     está bien como texto, cinco valores y todos consistentes
# unit_price  debería convertirse a número, quitando el signo y la coma
```

La comparación contra `"object"` y contra `"str"` cubre las dos versiones de pandas. Si solo
comparas contra una, el ejercicio funciona en tu máquina y falla en la de tu compañero.

### Ejercicio 5

```python
print("Rellenando con cero:", round(ventas["units"].fillna(0).mean(), 2))
print("Descartando:        ", round(ventas["units"].dropna().mean(), 2))
print("Dejándolos como están:", round(ventas["units"].mean(), 2))

print("\nRenglones que entran en cada cuenta:")
print("Rellenando con cero:", ventas["units"].fillna(0).count())
print("Descartando:        ", ventas["units"].dropna().count())

# Para "promedio de unidades por venta" va 16.12, o sea descartar o dejarlos.
# Rellenar con cero inventa once ventas de cero unidades que nunca ocurrieron,
# y arrastra el promedio hacia abajo por una razón que no está en los datos.
```

Salen 15.57 con ceros y 16.12 en los otros dos casos. La diferencia parece chica hasta que
la multiplicas por el volumen anual.

Fíjate en algo que sorprende: descartar y dejarlos como están dan el **mismo promedio**,
porque `mean` ya ignora los faltantes por su cuenta. Lo que sí cambia entre esos dos es el
conteo, 313 contra 313 aquí, pero en cuanto sumes o dividas por `len(df)` empiezan a
separarse. Esa es la razón por la que hay que decidirlo explícitamente en lugar de confiar
en lo que haga el método.

### Ejercicio 6

```python
CORRECTAS = ["North", "South", "Centre", "West"]

sucios = ~ventas["region"].isin(CORRECTAS)
print("Renglones con una región mal capturada:", sucios.sum())
print(ventas.loc[sucios, "region"].value_counts())

print("\nLo que el archivo cree:", (ventas["region"] == "North").sum())
print("Lo que de verdad es:  ",
      ventas["region"].str.strip().str.title().eq("North").sum())
```

Veinticuatro renglones traen una versión sucia. El norte real tiene 99 ventas y el archivo
reporta 75, así que un reporte hecho hoy le quita al norte una cuarta parte de su volumen y
se lo reparte a cuatro regiones fantasma.

`.str.strip().str.title()` es un adelanto de la sesión 15.2, y lo usamos aquí solo para
medir. Corregir el archivo es la clase que sigue.

### Ejercicio 7

```python
empleados = pd.read_csv("employees.csv")

print(empleados.shape)
print("\nÁreas:", empleados["area"].nunique())
print(empleados["area"].value_counts())

mas_alto = empleados["monthly_salary"].idxmax()
print("\nSalario más alto:", empleados["monthly_salary"].max())
print("Lo cobra:", empleados.loc[mas_alto, "employee_id"],
      "en", empleados.loc[mas_alto, "area"])
```

Cinco áreas, 120 personas, y el salario más alto es 82,700 de `E0003`.

Lo que hace útil este ejercicio es `idxmax`. Devuelve la **etiqueta del renglón** donde está
el máximo, no el máximo, y con esa etiqueta `.loc` te trae el renglón entero. Es el patrón
de "quién tiene el valor más alto", y lo vas a usar en cada reporte del semestre.

### Ejercicio 8

No hay solución publicada porque el archivo es distinto para cada quien. El diagnóstico se
califica sobre cuatro cosas: que estén los cuatro números pedidos, que nombres las columnas
que salieron texto, que digas de cada una si eso está bien, y que no hayas limpiado nada
todavía.